In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

jdbc_host = "dea.cgyi97rb4alr.us-east-1.rds.amazonaws.com"
jdbc_port = "5432"
jdbc_database = "dea_analytics_dev"

jdbc_url = (
    f"jdbc:postgresql://{jdbc_host}:{jdbc_port}/{jdbc_database}"
)

connection_properties = {
    "user": "student_user",
    "password": "DataEngineer12345",
    "driver": "org.postgresql.Driver"
}

CATALOG = "crm_ingestion"
BRONZE_SCHEMA = "bronze"

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS
{CATALOG}.{BRONZE_SCHEMA}
""")

print("✅ Connection loaded")

In [0]:
config_df = (
    spark.table("crm_ingestion.config.bronze_config")
         .filter("active = true")
)

display(config_df)

In [0]:
def load_postgres_table(
    source_table: str,
    target_table: str,
    incremental_column: str
):

    query = f"""
    (
        SELECT *
        FROM {source_table}
        WHERE DATE({incremental_column}) = CURRENT_DATE
    ) t
    """

    df = (
        spark.read
             .format("jdbc")
             .option("url", jdbc_url)
             .option("dbtable", query)
             .option("user", connection_properties["user"])
             .option("password", connection_properties["password"])
             .option("driver", connection_properties["driver"])
             .load()
    )

    (
        df.write
          .format("delta")
          .mode("append")
          .saveAsTable(target_table)
    )

    print(f"✅ Loaded {source_table}")

In [0]:
configs = config_df.collect()

for row in configs:

    load_postgres_table(
        source_table=row.source_table,
        target_table=row.target_table,
        incremental_column=row.incremental_column
    )

print("✅ Bronze ingestion complete")